# Importing and Validating the ZoomZoom (`sqlda`) Dataset
### Student guide | Neon PostgreSQL + DBeaver + `psql` | Companion to *SQL for Data Analytics*, Fourth Edition


**Primary sources:** Jun Shan et al., *SQL for Data Analytics*, 4th ed. (Packt, 2025), Chapter 1, Exercise 1.4 (pp. 23–24), *Introducing the sqlda database* (p. 25), Activity 1 (p. 26); Chapter 3, *Exchanging Data Using COPY* (pp. 49–64). [Companion Datasets](https://github.com/PacktPublishing/SQL-for-Data-Analytics-Fourth-Edition/tree/main/Datasets). 

## Learning outcomes and completion checklist

By the end you should be able to:

- [ ] Download and identify the `data.dump` format.
- [ ] Identify the target database (for example, `zoomzoom`) and confirm your SQL editor is using it.
- [ ] Prepare for ownership roles referenced by the source dump, as permitted by your Neon account.
- [ ] Choose the correct import path for the file format.
- [ ] Check the script execution log for `COPY` results and meaningful errors.
- [ ] List all imported tables and count their rows.
- [ ] Inspect column definitions, check NULLs, and inspect sales value ranges.
- [ ] Capture and submit the required row-count screenshot.


## 0. Before you begin - what the dataset contains

ZoomZoom is a fictional EV/scooter retailer used for customer, product, sales, campaign, and geographic analysis. 

**You need:** DBeaver, access to your Neon PostgreSQL instance, the course dataset, and a target database you can modify. 

> A Neon **connection** can expose multiple PostgreSQL databases. The database `neondb` is not automatically the same as a database named `zoomzoom`. Within a database, `public` is a **schema**. A toolbar showing `public@zoomzoom` indicates the `public` schema in the `zoomzoom` database. The SQL editor may still have a different connection context; verify it explicitly below.



## 1. Download the source file

1. Open the [textbook companion repository's Datasets folder](https://github.com/PacktPublishing/SQL-for-Data-Analytics-Fourth-Edition/tree/main/Datasets).
2. Download **`data.dump`** (use GitHub's file download / raw download option, not a saved HTML preview page).
3. Save it in a known local folder. You do not need to rename the extension.
4. If the download is compressed, extract it first. If it is already a `.dump` file, do not rename it to `.sql` just to make a tool accept it.
5. **Check its format** before choosing Restore versus SQL script execution.

**Simple format check:** Open the file in DBeaver's text editor or a plain-text editor. If you can read lines like `-- PostgreSQL database dump`, `SET ...`, `CREATE TABLE`, and `COPY public.<table> ... FROM stdin;`, it is a **plain-text SQL dump**. A PostgreSQL custom-format archive is binary and typically begins with the `PGDMP` signature. The extension `.dump` does not tell you the format.

## 2. Select the correct Neon database in DBeaver

1. Open **Database Navigator / Connections**.
2. Expand your Neon PostgreSQL connection → **Databases**.
3. Select your intended target database (for example, **`zoomzoom`**). If it does not exist, create it.
4. Right-click that database → **SQL Editor → New SQL Script** (wording may vary).
5. Run the following check and record its output.

**Important:** Selecting `zoomzoom` in the left Navigator does not automatically retarget an SQL editor that was opened earlier for `neondb`.

```sql
SELECT current_database() AS connected_database,
       current_schema() AS active_schema,
       current_user AS connected_user,
       version() AS postgresql_version;
```

**Expected:** `connected_database` must be the database you intend to import into. If it says `neondb` while you intend `zoomzoom`, stop and open a new SQL editor on the correct database. **Do not import into the wrong database and then validate another one.**

## 3. Prepare the owner roles 

The source dump references roles such as `azure_pg_admin` and `junshan`. In your case, create placeholder roles **before** importing. Run the commands in an editor connected to your Neon instance.

> Note: Creating a role and being allowed to `SET ROLE` to it are separate permissions.

```sql
CREATE ROLE azure_pg_admin WITH LOGIN PASSWORD 'REPLACE_WITH_UNIQUE_PASSWORD_1';
CREATE ROLE junshan WITH LOGIN PASSWORD 'REPLACE_WITH_UNIQUE_PASSWORD_2';
```

**Important:** PostgreSQL ownership commands like `ALTER TABLE ... OWNER TO junshan` can still fail with `must be able to SET ROLE "junshan"` even after the role exists. The error matters, but it is not by itself proof that the data import failed.

## 4. Choose the right import method

### Route A - Use DBeaver Restore **only** for a supported PostgreSQL archive

- Right-click the **target database** → **Tools → Restore**.
- Select the file, and configure the PostgreSQL local client if prompted.
- If `pg_restore` reports **`did not find magic string in file header`** (or **`input file appears to be a text format dump`**), **stop**. The file is not a compatible custom-format archive for this workflow. Use **Route B**.

### Route B - Plain-text `data.dump`: execute it with `psql` through DBeaver

1. Right-click the **target database** (e.g., `zoomzoom`).
2. Select **Tools → Execute script** (DBeaver's PostgreSQL native script execution, wording may vary).
3. In **Input**, browse to the actual local `data.dump` file. Check the full path: it must point to `data.dump`, **not** another sample `.sql` file.
4. Verify the selected target database and the connection credentials.
5. If prompted for a **Local Client**, choose/install a PostgreSQL client that includes `psql`.
6. Click **Start**, then wait for the task to finish. Review the complete log, including the **first** error.

**Why this route:** The dump file contained PostgreSQL `COPY ... FROM stdin` data blocks. `psql` understands this script format. **Do not** execute a plain-text dump containing `COPY ... FROM stdin` by simply selecting all text and pressing the ordinary single-statement execute button in a JDBC SQL editor. That is not a reliable equivalent to running it through `psql`.

**Alternative:** Run `psql` from a terminal using the correct connection parameters and `-f /path/to/data.dump`. 

**Why the SQL-editor fallback sometimes fails:** A dump may contain `COPY ... FROM stdin` followed by raw rows and a `\.` terminator. An ordinary SQL editor may parse this differently from `psql`. If DBeaver's SQL-script execution produces empty tables, stop and use a native `psql` execution path; do not assume that creating the tables means the import succeeded.

## 5. Read the execution log

In a successful data-loading stage, the PostgreSQL log prints results like:

```text
COPY 44533
COPY 0
COPY 50000
COPY 50000
COPY 32
```

`COPY 50000` means that operation copied **50,000 rows**. `COPY 0` means **zero rows** were copied for that particular table; this is not necessarily an error. 

**What to look for:**

- `CREATE TABLE` / `CREATE ...` indicates objects were defined, **not** that rows were loaded.
- `COPY <number>` indicates data loading took place for that operation.
- `ERROR: must be able to SET ROLE "junshan"` points to an ownership/permission issue. Inspect whether subsequent `COPY` operations succeeded.
- `ERROR: relation ... already exists` means you may be rerunning DDL against an existing schema.
- `ERROR: duplicate key value violates unique constraint` can indicate a partial previous import or rerun.
- A blank log **while running** is not a completed import. Wait for termination; inspect the final status and log.

**Do not click Start repeatedly.** A full replay into a partially populated database may create duplicate rows or conflicts.

## 6. Validation 1 - confirm the 13 tables exist

Run this in a new SQL editor connected to the **same target database**. Compare names with your dump's DDL. This should return 13 tables, including:

`closest_dealerships`, `countries`, `customer_sales`, `customer_sales_copy`, `customer_survey`, `customers`, `dealerships`, `emails`, `products`, `public_transportation_by_zip`, `sales`, `salespeople`, `top_cities_data`.


```sql
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
  AND table_type = 'BASE TABLE'
ORDER BY table_name;
```

**Interpretation:** Seeing 13 table names is a good first check, but **it does not establish that the data imported**. Proceed to the row counts.

## 7. Validation 2 - row counts for **all 13** tables 

Copy this query into DBeaver, run it against the target database; which will return **all 13 table names and counts**. 

```sql
SELECT 'closest_dealerships' AS table_name, COUNT(*) AS row_count FROM public.closest_dealerships
UNION ALL SELECT 'countries', COUNT(*) FROM public.countries
UNION ALL SELECT 'customer_sales', COUNT(*) FROM public.customer_sales
UNION ALL SELECT 'customer_sales_copy', COUNT(*) FROM public.customer_sales_copy
UNION ALL SELECT 'customer_survey', COUNT(*) FROM public.customer_survey
UNION ALL SELECT 'customers', COUNT(*) FROM public.customers
UNION ALL SELECT 'dealerships', COUNT(*) FROM public.dealerships
UNION ALL SELECT 'emails', COUNT(*) FROM public.emails
UNION ALL SELECT 'products', COUNT(*) FROM public.products
UNION ALL SELECT 'public_transportation_by_zip', COUNT(*) FROM public.public_transportation_by_zip
UNION ALL SELECT 'sales', COUNT(*) FROM public.sales
UNION ALL SELECT 'salespeople', COUNT(*) FROM public.salespeople
UNION ALL SELECT 'top_cities_data', COUNT(*) FROM public.top_cities_data
ORDER BY table_name;
```

## 8. Validation 3 - inspect the `customers` table columns

Check that the expected columns and data types are present. 

```sql
SELECT column_name, data_type, is_nullable
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name = 'customers'
ORDER BY ordinal_position;
```

## 9. Validation 4- check for missing customer IDs

Use the actual ID column from Validation 3. 

```sql
SELECT COUNT(*) AS null_count
FROM public.customers
WHERE customer_id IS NULL;
```

**Interpretation:** `0` means none of the **existing** customer rows has a NULL `customer_id`. An empty table also returns `0` here, so **always combine this test with the row-count check**. The column's `is_nullable = YES` is not inconsistent with a zero NULL count.

## 10. Validation 5 - inspect sales date and amount ranges

The following query uses column names from the database. Use `information_schema.columns` first if your `sales` schema differs.

```sql
SELECT MIN(sales_transaction_date) AS earliest_date,
       MAX(sales_transaction_date) AS latest_date,
       MIN(sales_amount) AS min_amount,
       MAX(sales_amount) AS max_amount,
       ROUND(AVG(sales_amount)::numeric, 2) AS avg_amount
FROM public.sales;
```

## 11. Troubleshooting decision tree

**A. `pg_restore: did not find magic string in file header`**

**Likely cause:** You gave `pg_restore` a plain-text SQL dump. **Action:** Open the file as text to confirm, then use **Tools → Execute script** / `psql`, not Restore. Do not repeatedly retry Restore on the same text file.

**B. All 13 tables exist but all 13 row counts are 0**

**Likely causes:** The DDL ran but the `COPY ... FROM stdin` sections did not; the script was executed with the wrong tool; execution stopped early; or validation is targeting a different database. **Action:** Run `SELECT current_database();`, inspect the `COPY` log, and verify the actual `data.dump` path. Use native **Execute script** with `psql`. Do not start by recreating tables.

**C. `must be able to SET ROLE "junshan"` / owner-change errors**

**Likely cause:** The executing role lacks the privileges required to change ownership. **Action:** Confirm placeholder roles exist and inspect whether the `COPY` stages still succeeded. Do not grant broad privileges or change server roles indiscriminately.

**D. `role "junshan" does not exist` or `role "azure_pg_admin" does not exist`**

**Likely cause:** The source dump references roles absent from the destination. **Action:** Complete the assignment's placeholder-role step, if permitted, then retry on an appropriate clean target.

**E. `relation ... already exists`**

**Likely cause:** The schema was already created during an earlier attempt. **Action:** Check table counts and current database. For a clean rerun, prefer a **new empty practice database** if allowed; do not blindly drop the `public` schema or production objects.

**F. `duplicate key value violates unique constraint`**

**Likely cause:** A previous run already loaded some data, or the same data is being loaded twice. **Action:** Stop, inspect counts, and decide whether an clean restore is needed. Avoid appending the full dump again.

**G. `COPY 0` or `countries = 0`**

**Meaning:** Zero records copied for that operation/table. **Action:** Compare with the dump's contents and documentation. In this case, `countries` was empty even though the main dataset was loaded. A zero for **one** table is not the same as zeros for **all** tables.

**H. `permission denied for schema public` / cannot create objects**

**Likely cause:** The connected account lacks CREATE rights in that database/schema. **Action:** Confirm `current_user`, target database, and the provided Neon permissions. Ask for the correct database role rather than trying privileged changes.

**I. Script runs but no rows appear / log is blank**

**Action:** Wait for completion; check the task's final status and first error, not just the progress bar. Confirm the input file path points to `data.dump` and that the database is correct. Run counts again in a new editor on the same database.

**J. Red squiggles under table names in DBeaver**

**Possible cause:** SQL-editor metadata cache or wrong active connection. **Action:** Confirm with an actual `SELECT` execution and refresh the Navigator (F5). A red underline alone is not proof that the table is missing.


## 12. Safe recovery workflow for a partial import

1. **Stop re-running the dump.** Record the first error from the execution log.
2. Run `SELECT current_database(), current_user;` and confirm the intended target.
3. Run the 13-table count query. Note which tables, if any, contain rows.
4. Inspect the dump format and whether it contains `COPY public.` blocks.
5. Decide whether the import can be repaired without recreating objects. For a complicated partial restore, **new empty database** is often easier and safer than manually dropping tables.
6. If starting fresh, confirm the database is empty **before** executing the script once through the correct native `psql` workflow.
7. Re-run all five validation steps and capture a new screenshot.

**Never run `DROP DATABASE`, `DROP SCHEMA public CASCADE`, or broad deletion commands against a shared or uncertain database.** A separate clean database is appropriate only when you have permission and are sure it contains no work you need to keep.

## Textbook verification exercise -inspect real product rows

Use this simple check after importing:

```sql
SELECT * FROM public.products LIMIT 5;
```

**Expected interpretation:** You should see real product records and recognizable fields such as `product_id`, `model`, `year`, `product_type`, and `base_msrp`. The exact first five rows may differ; the textbook itself cautions that the records can vary. A table that exists but returns **zero rows** has not passed this smoke test.
.